<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS11/Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Buatlah model SVM dengan menggunakan data voice.csv dengan ketentuan,

- Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.

  - Gunakan model dengan kernel linier.

  - Gunakan model dengan kernel polynomial.

  - Gunakan model dengan kernel RBF.

- Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load Dataset
df = pd.read_csv('voice.csv')

# Separasi Fitur (X) dan Label (y)
X = df.drop(columns=['label'])
y = df['label']

# Encode label string ('male', 'female') menjadi nilai numerik
le = LabelEncoder()
y = le.fit_transform(y)

# Standardisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Pengujian Eksperimen
split_ratios = [0.3, 0.2] # 0.3 untuk 70:30, 0.2 untuk 80:20
kernels = ['linear', 'poly', 'rbf']

results = []

for test_size in split_ratios:
    ratio_label = "70:30" if test_size == 0.3 else "80:20"

    # Split Data
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )

    for kernel in kernels:
        # Inisialisasi & Fitting Model SVM
        model = SVC(kernel=kernel)
        model.fit(X_train, y_train)

        # Prediksi & Evaluasi Akurasi
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Rasio Split': ratio_label,
            'Kernel': kernel.upper(),
            'Akurasi': round(acc, 4)
        })

# 3. Tabulasi Performansi
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Rasio Split Kernel  Akurasi
      70:30 LINEAR   0.9706
      70:30   POLY   0.9569
      70:30    RBF   0.9811
      80:20 LINEAR   0.9763
      80:20   POLY   0.9685
      80:20    RBF   0.9826


Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!

In [3]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.image as mpimg
import zipfile
import os
from google.colab import files
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# --- 0. Upload & Ekstrak Dataset Otomatis ---
if not os.path.exists("images/training"):
    if not os.path.exists("images.zip"):
        print("Silakan pilih dan upload file 'images.zip':")
        uploaded = files.upload()

    if os.path.exists("images.zip"):
        with zipfile.ZipFile("images.zip", 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Dataset berhasil diekstrak!")

# --- 1. Load Dataset ---
def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')
    img_list = []
    for dir_path in dirs:
        if dir_path.is_dir():
            label = dir_path.name
            for file in dir_path.glob('*.jpg'):
                img = mpimg.imread(file)
                if img is not None:
                    img_list.append((img, label))
    return img_list

train_dir = "images/training/"
train_img = load_dataset(train_dir)
print(f"Jumlah gambar berhasil dimuat: {len(train_img)}")

# --- 2. Ekstraksi Fitur Histogram Citra ---
def extract_histogram(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [8], [0, 180])
    hist_s = cv2.calcHist([hsv], [1], None, [8], [0, 256])
    hist_v = cv2.calcHist([hsv], [2], None, [8], [0, 256])

    hist = np.concatenate([hist_h, hist_s, hist_v]).flatten()
    total = np.sum(hist)
    if total > 0:
        hist = hist / total
    return hist

# --- 3. Ekstraksi Fitur & Labeling ---
X = []
y = []

for img, label in train_img:
    features = extract_histogram(img)
    X.append(features)
    y.append(label)

X = np.array(X)
y = np.array(y)

# --- 4. Split Data (Rasio 80:20) & Standarisasi ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 5. Model SVM RBF Baseline ---
model_rbf = SVC(kernel='rbf', random_state=42)
model_rbf.fit(X_train_scaled, y_train)

y_pred = model_rbf.predict(X_test_scaled)
acc_baseline = accuracy_score(y_test, y_pred)
print(f"Akurasi Baseline RBF: {acc_baseline * 100:.2f}%")

# --- 6. Hyperparameter Tuning (GridSearchCV) ---
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

# --- 7. Evaluasi Model Terbaik ---
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)
acc_best = accuracy_score(y_test, y_pred_best)

print("\n--- Hasil Tuning ---")
print(f"Parameter Terbaik : {grid_search.best_params_}")
print(f"Akurasi Setelah Tuning: {acc_best * 100:.2f}%")

Silakan pilih dan upload file 'images.zip':


Saving images.zip to images.zip
Dataset berhasil diekstrak!
Jumlah gambar berhasil dimuat: 240
Akurasi Baseline RBF: 100.00%

--- Hasil Tuning ---
Parameter Terbaik : {'C': 1, 'gamma': 'scale'}
Akurasi Setelah Tuning: 100.00%
